# IK MLP Training

In [1]:
import sys
try:
    import google.colab

    from google.colab import drive
    drive.mount('/content/drive')
    !git clone https://github.com/max15189/InverseKinematics.git

    sys.path.insert(0, '/content/InverseKinematics')
    IN_COLAB = True

except ImportError:
    IN_COLAB = False

    sys.path.insert(0, r"c:\Users\max\InverseKinematics")



Mounted at /content/drive
Cloning into 'InverseKinematics'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 115 (delta 49), reused 87 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 1.23 MiB | 21.76 MiB/s, done.
Resolving deltas: 100% (49/49), done.


False

## Config

In [8]:
SAVE_DATA = "/content/drive/MyDrive/inverse_kinematics/new_dataset" if IN_COLAB else "G:/My Drive/inverse_kinematics/new_dataset"
SAVE_MODEL = f"{SAVE_DATA}/mlp_ik_512.pt"

BATCH_SIZE = 512
EPOCHS     = 200
LR         = 1e-3
PATIENCE   = 30

## Imports

In [4]:

import torch
from torch.utils.data import DataLoader

from ik.data.dataset import IKDataset
from ik.model.mlp import MLP
from ik.model.training import run_training, evaluate_and_return_loss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

device: cuda


## Load Data

In [5]:
train_ds = IKDataset("train", SAVE_DATA)
val_ds   = IKDataset("val",   SAVE_DATA,
                     MinMax_X=train_ds.MinMax_X,
                     MinMax_Y=train_ds.MinMax_Y)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"train: {len(train_ds):,} samples")
print(f"val:   {len(val_ds):,} samples")
print(f"input dim: {train_ds.X.shape[1]}  (R6[6] + P[3] + q_init[6])")
print(f"output dim: {train_ds.y.shape[1]}  (q_target[6])")

train: 1,440,000 samples
val:   180,000 samples
input dim: 15  (R6[6] + P[3] + q_init[6])
output dim: 6  (q_target[6])


## Model

In [9]:
model = MLP(input_dim=15, hidden_dim=512, output_dim=6, n_layers=4).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

parameters: 799,238
MLP(
  (net): Sequential(
    (0): Linear(in_features=15, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=512, out_features=6, bias=True)
  )
)


## Train

In [10]:
model = run_training(
    model,
    train_loader,
    val_loader,
    MinMax_Y=train_ds.MinMax_Y,
    device=DEVICE,
    save_path=SAVE_MODEL,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
)

epoch   1 | train mse 0.0110 (pos 0.0838 rot 0.4733) | val mse 0.0058 (pos 0.0528 rot 0.3246) | lr 1.00e-03
epoch   2 | train mse 0.0044 (pos 0.0452 rot 0.2717) | val mse 0.0038 (pos 0.0407 rot 0.2354) | lr 1.00e-03
epoch   3 | train mse 0.0034 (pos 0.0385 rot 0.2250) | val mse 0.0032 (pos 0.0366 rot 0.2115) | lr 1.00e-03
epoch   4 | train mse 0.0029 (pos 0.0350 rot 0.2025) | val mse 0.0027 (pos 0.0336 rot 0.1869) | lr 1.00e-03
epoch   5 | train mse 0.0025 (pos 0.0326 rot 0.1873) | val mse 0.0025 (pos 0.0331 rot 0.1836) | lr 1.00e-03
epoch   6 | train mse 0.0023 (pos 0.0309 rot 0.1771) | val mse 0.0024 (pos 0.0311 rot 0.1764) | lr 1.00e-03
epoch   7 | train mse 0.0021 (pos 0.0297 rot 0.1692) | val mse 0.0021 (pos 0.0287 rot 0.1654) | lr 1.00e-03
epoch   8 | train mse 0.0020 (pos 0.0286 rot 0.1631) | val mse 0.0020 (pos 0.0269 rot 0.1588) | lr 1.00e-03
epoch   9 | train mse 0.0018 (pos 0.0278 rot 0.1585) | val mse 0.0019 (pos 0.0280 rot 0.1656) | lr 1.00e-03
epoch  10 | train mse 0.0018

## Evaluate on Val Set

In [ ]:
mse_losses, pos_losses, rot_losses = evaluate_and_return_loss(
    model, val_loader, DEVICE, MinMax_Y=train_ds.MinMax_Y
)

In [ ]:
model.load_state_dict(torch.load(SAVE_MODEL))

<All keys matched successfully>

In [ ]:
train_ds.MinMax_Y

[array([-3.1415927, -1.7627826, -1.7627826, -3.1415927, -1.8675023,
        -3.1415927], dtype=float32),
 array([3.1415927, 1.7627826, 1.6057029, 3.1415927, 2.268928 , 3.1415927],
       dtype=float32)]

In [ ]:
from ik.model.training import _denorm_q,_to_tensors
test_ds=IKDataset("test",SAVE_DATA,train_ds.MinMax_X,train_ds.MinMax_Y)
Y_min,Y_max=_to_tensors(test_ds.MinMax_Y,DEVICE)
q_predict=model(torch.tensor(test_ds.X))
q_predict=_denorm_q(q_predict,Y_min,Y_max)
q_predict=q_predict.cpu().detach().numpy()

IKinSpace from moderin roboitcs library with iteration in the return

In [ ]:
!pip install modern-robotics
import numpy as np
import modern_robotics as mr
def IKinSpace_with_iters(Slist, M, T, thetalist0, eomg, ev):
    thetalist = np.array(thetalist0).copy()
    i = 0
    maxiterations = 50
    Tsb = mr.FKinSpace(M,Slist, thetalist)
    Vs = np.dot(mr.Adjoint(Tsb),mr.se3ToVec(mr.MatrixLog6(np.dot(mr.TransInv(Tsb), T))))
    err = np.linalg.norm([Vs[0], Vs[1], Vs[2]]) > eomg or np.linalg.norm([Vs[3], Vs[4], Vs[5]]) >  ev
    while err and i < maxiterations:
        thetalist = thetalist + np.dot(np.linalg.pinv(mr.JacobianSpace(Slist,thetalist)), Vs)
        i = i + 1
        Tsb = mr.FKinSpace(M, Slist, thetalist)
        Vs = np.dot(mr.Adjoint(Tsb), \
                    mr.se3ToVec(mr.MatrixLog6(np.dot(mr.TransInv(Tsb), T))))
        err = np.linalg.norm([Vs[0], Vs[1], Vs[2]]) > eomg \
              or np.linalg.norm([Vs[3], Vs[4], Vs[5]]) > ev
    return (thetalist, not err,i)

In [ ]:
from ik.kinematics.fk import FK,FK_batch_full,_Slist_np,_M_HOME_np

i=1
IKinSpace_with_iters(_Slist_np,_M_HOME_np,FK(test_ds.q_target[i]),q_predict[i],0.001,0.001)

(array([-1.44960048, -1.1817319 ,  0.04990294,  1.92684414,  0.15471801,
         1.23075421]),
 True,
 1)

In [ ]:
import numpy as np
! pip install tdqm
from tqdm import tqdm
iterations_list = []
convergence_list = []

for i in tqdm(range(len(q_predict))): # Iterate through all predicted configurations
    # Use the true target pose for the IKinSpace_with_iters function
    target_T = FK(test_ds.q_target[i])
    initial_q = q_predict[i]

    # Call IKinSpace_with_iters
    thetalist_out, converged, num_iters = IKinSpace_with_iters(_Slist_np, _M_HOME_np, target_T, initial_q, 0.001, 0.001)

    iterations_list.append(num_iters)
    convergence_list.append(converged)


average_iterations = np.mean(iterations_list)
percentage_convergence = np.sum(convergence_list) / len(convergence_list) * 100

print(f"Average iterations: {average_iterations:.2f}")
print(f"Percentage of convergence: {percentage_convergence:.2f}%")

100%|██████████| 30000/30000 [01:46<00:00, 281.67it/s]

Average iterations: 1.50
Percentage of convergence: 100.00%
